# Leereenheid 4.1: Data-skoonmaak

## Hanteer datakwaliteitskwessies en pas toepaslike tegnieke toe om data skoon te maak

### Gevallestudie: Boland Meubels & Toestelle

Pieter van der Merwe het sy verkoopsdata van drie verskillende kassastelsels ontvang. Voor hy kan begin met sy analise, moet hy eers die data skoonmaak.

In [ ]:
# Laai nodige biblioteke
import pandas as pd
import numpy as np

## Stap 1: Laai die rou data

Ons begin deur die rou verkoopsdata te laai.

In [ ]:
# Laai die rou verkoopsdata
url = 'https://raw.githubusercontent.com/aby-akademia/NGRDA150-2026/main/datastelle/le4_boland_meubels_verkope_rou.csv'
data = pd.read_csv(url)

# Kyk na die eerste paar rye
data.head()

In [ ]:
# Kyk na basiese inligting oor die datastel
print(f"Aantal rye: {len(data)}")
print(f"Aantal kolomme: {len(data.columns)}")
print("\nKolomme:")
print(data.columns.tolist())

## Stap 2: Identifiseer datakwaliteitsprobleme

Kom ons kyk watter probleme in die data is.

In [ ]:
# Kyk vir ontbrekende waardes
print("Ontbrekende waardes per kolom:")
print(data.isnull().sum())

In [ ]:
# Kyk vir duplikate transaksies
duplikate = data.duplicated(subset=['transaksie_id'], keep=False)
print(f"Aantal duplikate transaksies: {duplikate.sum()}")

# Wys die duplikate
if duplikate.sum() > 0:
    print("\nVoorbeeld van duplikate:")
    print(data[duplikate].sort_values('transaksie_id').head(10))

In [ ]:
# Kyk na unieke winkelname (moet net 3 wees: Stellenbosch, Paarl, Worcester)
print("Unieke winkelname:")
print(data['winkel_naam'].unique())

## Stap 3: Hanteer ontbrekende waardes

Ons moet besluit wat om te doen met ontbrekende waardes.

In [ ]:
# Maak 'n kopie van die data
data_skoon = data.copy()

# Strategie vir ontbrekende waardes:
# - kosprys: vul in met mediaan (meer stabiel as gemiddelde)
# - klient_id: verwyder die rye (kan nie verkope sonder klient analiseer nie)
# - produk: verwyder die rye (kan nie verkope sonder produk analiseer nie)

print("Voor skoonmaak:")
print(f"Aantal rye: {len(data_skoon)}")

# Vul ontbrekende kosprys in met mediaan
mediaan_kosprys = data_skoon['kosprys'].median()
data_skoon['kosprys'] = data_skoon['kosprys'].fillna(mediaan_kosprys)

# Verwyder rye waar klient_id of produk ontbreek
data_skoon = data_skoon.dropna(subset=['klient_id', 'produk'])

print(f"\nNa skoonmaak:")
print(f"Aantal rye: {len(data_skoon)}")
print(f"Ontbrekende waardes: {data_skoon.isnull().sum().sum()}")

## Stap 4: Verwyder duplikate

Ons behou net die eerste voorkoms van elke transaksie.

In [ ]:
print("Voor duplikaat verwydering:")
print(f"Aantal rye: {len(data_skoon)}")

# Verwyder duplikate (hou eerste voorkoms)
data_skoon = data_skoon.drop_duplicates(subset=['transaksie_id'], keep='first')

print(f"\nNa duplikaat verwydering:")
print(f"Aantal rye: {len(data_skoon)}")

## Stap 5: Standaardiseer data formate

Ons moet verseker dat al die data in konsekwente formate is.

In [ ]:
# Standaardiseer winkelname (verwyder spasies, maak hoofletters)
data_skoon['winkel_naam'] = data_skoon['winkel_naam'].str.strip().str.title()

print("Gestandaardiseerde winkelname:")
print(data_skoon['winkel_naam'].unique())

In [ ]:
# Standaardiseer verkoopsprys (sommige is strings met R en kommas)
# Eers, skakel alles na strings
data_skoon['verkoopsprys'] = data_skoon['verkoopsprys'].astype(str)

# Verwyder R en kommas
data_skoon['verkoopsprys'] = data_skoon['verkoopsprys'].str.replace('R', '', regex=False)
data_skoon['verkoopsprys'] = data_skoon['verkoopsprys'].str.replace(',', '', regex=False)

# Skakel na numeries
data_skoon['verkoopsprys'] = pd.to_numeric(data_skoon['verkoopsprys'], errors='coerce')

print("Verkoopsprys is nou numeries:")
print(data_skoon['verkoopsprys'].head())

In [ ]:
# Skakel datum na datetime formaat
data_skoon['datum'] = pd.to_datetime(data_skoon['datum'])

print("Datum is nou datetime formaat:")
print(data_skoon['datum'].head())

## Stap 6: Identifiseer en hanteer uitskieters

Ons kyk vir onrealistiese waardes.

In [ ]:
# Kyk vir negatiewe kosprys
negatiewe_kosprys = data_skoon['kosprys'] < 0
print(f"Aantal transaksies met negatiewe kosprys: {negatiewe_kosprys.sum()}")

# Maak negatiewe kosprys positief (aanvaar dit is tikfoute)
data_skoon.loc[negatiewe_kosprys, 'kosprys'] = data_skoon.loc[negatiewe_kosprys, 'kosprys'].abs()

In [ ]:
# Kyk vir onrealistiese hoeveelhede
print("Beskrywende statistiek vir hoeveelheid:")
print(data_skoon['hoeveelheid'].describe())

# Verwyder transaksies met hoeveelheid <= 0 of > 10 (onrealisties vir meubels)
voor_filter = len(data_skoon)
data_skoon = data_skoon[(data_skoon['hoeveelheid'] > 0) & (data_skoon['hoeveelheid'] <= 10)]
na_filter = len(data_skoon)

print(f"\nVerwyder {voor_filter - na_filter} transaksies met onrealistiese hoeveelhede")

## Stap 7: Finale validasie

Kom ons verseker die data is nou skoon.

In [ ]:
print("FINALE DATA KWALITEIT VERSLAG")
print("=" * 50)
print(f"Aantal transaksies: {len(data_skoon)}")
print(f"Ontbrekende waardes: {data_skoon.isnull().sum().sum()}")
print(f"Duplikate transaksies: {data_skoon.duplicated(subset=['transaksie_id']).sum()}")
print(f"Unieke winkels: {data_skoon['winkel_naam'].nunique()}")
print(f"Unieke produkte: {data_skoon['produk'].nunique()}")
print(f"Datum reeks: {data_skoon['datum'].min()} tot {data_skoon['datum'].max()}")

In [ ]:
# Wys die skoon data
data_skoon.head(10)

## Stap 8: Stoor die skoon data

Nou kan ons die skoon data stoor vir gebruik in volgende leereenheid.

In [ ]:
# Stoor die skoon data
data_skoon.to_csv('boland_meubels_verkope_skoon.csv', index=False)
print("Skoon data gestoor as: boland_meubels_verkope_skoon.csv")

## Opsomming

In hierdie notaboek het ons:
1. Die rou verkoopsdata gelaai
2. Datakwaliteitsprobleme geïdentifiseer (ontbrekende waardes, duplikate, inkonsekwente formate)
3. Ontbrekende waardes hanteer (vul in of verwyder)
4. Duplikate verwyder
5. Data formate gestandaardiseer (winkelname, pryse, datums)
6. Uitskieters geïdentifiseer en hanteer (negatiewe pryse, onrealistiese hoeveelhede)
7. Die skoon data gestoor vir verdere analise

Pieter se data is nou gereed vir transformasie en analise in LU 4.2 en verder.